In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

kernel = "rbf"
nu_list = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
gamma_list = ["scale", "auto", 0.01, 0.05, 0.1, 0.5, 1.0]

data_path = Path("../data_processed/NAB/industrial_machine_features_labeled.csv")
results_root = Path("../results/OCSVM_NAB")
results_root.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path)

feature_cols = ["value", "diff1", "roll_mean", "roll_std", "roll_min", "roll_max"]
required_cols = feature_cols + ["y_true"]
df = df.dropna(subset=required_cols).reset_index(drop=True)

X = df[feature_cols].copy()
y_true = df["y_true"].astype(int)

X_normal = X[y_true == 0].copy()

scaler = StandardScaler()

X_train = pd.DataFrame(
    scaler.fit_transform(X_normal),
    columns=feature_cols,
    index=X_normal.index
)

X_test = pd.DataFrame(
    scaler.transform(X),
    columns=feature_cols,
    index=X.index
)

def gamma_to_str(gamma):
    if isinstance(gamma, str):
        return gamma
    return str(gamma).replace(".", "_")

def nu_to_str(nu):
    return str(nu).replace(".", "_")

all_results = []

for gamma in gamma_list:
    gamma_dir = results_root / f"gamma_{gamma_to_str(gamma)}"
    gamma_dir.mkdir(parents=True, exist_ok=True)

    gamma_results = []

    for nu in nu_list:
        exp_name = f"kernel_{kernel}_nu_{nu_to_str(nu)}"
        exp_dir = gamma_dir / exp_name
        exp_dir.mkdir(parents=True, exist_ok=True)

        model = OneClassSVM(
            kernel=kernel,
            nu=nu,
            gamma=gamma
        )

        model.fit(X_train)

        df_temp = df.copy()

        pred_raw = model.predict(X_test)
        decision_scores = model.decision_function(X_test)
        score_samples = model.score_samples(X_test)

        df_temp["ocsvm_pred_raw"] = pred_raw
        df_temp["ocsvm_decision"] = decision_scores
        df_temp["ocsvm_score"] = score_samples
        df_temp["is_anomaly"] = (df_temp["ocsvm_pred_raw"] == -1).astype(int)

        y_pred = df_temp["is_anomaly"]

        cm = confusion_matrix(y_true, y_pred)

        accuracy = accuracy_score(y_true, y_pred) * 100
        precision = precision_score(y_true, y_pred, zero_division=0) * 100
        recall = recall_score(y_true, y_pred, zero_division=0) * 100
        f1 = f1_score(y_true, y_pred, zero_division=0) * 100

        result_row = {
            "experiment": exp_name,
            "kernel": kernel,
            "nu": nu,
            "gamma": gamma,
            "accuracy_%": round(accuracy, 2),
            "precision_%": round(precision, 2),
            "recall_%": round(recall, 2),
            "f1_%": round(f1, 2)
        }

        all_results.append(result_row)
        gamma_results.append(result_row)

        df_temp.to_csv(exp_dir / "predictions.csv", index=False)

        metrics_df = pd.DataFrame({
            "Metrika": [
                "Presnosť (Accuracy)",
                "Precíznosť (Precision)",
                "Citlivosť (Recall)",
                "F1-skóre"
            ],
            "Hodnota (%)": [
                round(accuracy, 2),
                round(precision, 2),
                round(recall, 2),
                round(f1, 2)
            ]
        })

        fig, ax = plt.subplots(figsize=(6, 2))
        ax.axis("off")

        table = ax.table(
            cellText=metrics_df.values,
            colLabels=metrics_df.columns,
            loc="center",
            cellLoc="center"
        )

        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 2)

        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_text_props(weight="bold")
                cell.set_facecolor("#dce6f1")
            else:
                cell.set_facecolor("#f9f9f9")

        plt.title("Výsledné metriky modelu One-Class SVM – NAB", fontsize=12, pad=20)
        plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title("Matica zámen (Confusion Matrix)")
        plt.colorbar()

        classes = ["Normálne", "Anomálie"]
        tick_marks = np.arange(len(classes))

        plt.xticks(tick_marks, classes)
        plt.yticks(tick_marks, classes)

        threshold = cm.max() / 2.0 if cm.max() > 0 else 0.5

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                color = "black" if cm[i, j] > threshold else "white"
                plt.text(
                    j, i, cm[i, j],
                    ha="center", va="center",
                    color=color
                )

        plt.xlabel("Predikované triedy")
        plt.ylabel("Skutočné triedy")
        plt.tight_layout()
        plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close()

        if "timestamp" in df_temp.columns:
            x_axis = pd.to_datetime(df_temp["timestamp"])
            x_label = "Čas"
        else:
            x_axis = np.arange(len(df_temp))
            x_label = "Index"

        plt.figure(figsize=(14, 5))
        plt.plot(x_axis, df_temp["value"], linewidth=1)

        anomaly_idx = df_temp["is_anomaly"] == 1
        plt.scatter(
            np.array(x_axis)[anomaly_idx],
            df_temp.loc[anomaly_idx, "value"],
            s=18
        )

        plt.title(f"{exp_name}, gamma={gamma_to_str(gamma)}")
        plt.xlabel(x_label)
        plt.ylabel("Hodnota")
        plt.tight_layout()
        plt.savefig(exp_dir / "anomaly_plot.png", dpi=300, bbox_inches="tight")
        plt.close()

    gamma_results_df = pd.DataFrame(gamma_results)
    gamma_results_df = gamma_results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
    gamma_results_df.to_csv(gamma_dir / "summary_results.csv", index=False)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
results_df.to_csv(results_root / "summary_results.csv", index=False)

print("Done")

Done
